# CSD402 Capstone Option A — Colab runner
1. **Runtime → Change runtime type → T4 GPU**. 2. Upload the project zip and your dataset zip. 3. Run cells in order (each stage saves to `outputs/` and resumes).

In [ ]:
from google.colab import files
up = files.upload()   # CSD402_Capstone_OptionA.zip
!unzip -q -o CSD402_Capstone_OptionA.zip
%cd CSD402_Capstone_OptionA
!pip install -q -r requirements.txt

In [ ]:
# upload the dataset zip (or mount Drive) and unzip into data/raw
up = files.upload()
!mkdir -p data/raw && for z in *.zip; do [ "$z" != "CSD402_Capstone_OptionA.zip" ] && unzip -q -o "$z" -d data/raw; done
!python run.py inspect --data-root data/raw

In [ ]:
!python run.py prepare --data-root data/raw --minority defect --minority-keep 0.10   # add --label-map ... if needed

In [ ]:
!python run.py tune

In [ ]:
!python run.py gan --gan-epochs 300

In [ ]:
!python run.py arms --seeds 0 1 2

In [ ]:
# HELD-OUT TEST: once, after all decisions are final
!python run.py test

In [ ]:
!python run.py gradcam --arm gan
!python run.py export --arm gan

In [ ]:
import pandas as pd
from IPython.display import display, Image
for f in ['test_results_mean_std','threshold_tuning_target_recall','cost_comparison']: print(f); display(pd.read_csv(f'outputs/{f}.csv'))
print(open('outputs/bootstrap_minority_recall_difference.json').read())
for f in ['gan_samples','gan_losses','learning_curves','minority_recall','confusion_matrices','gradcam_gan']: display(Image(f'outputs/figures/{f}.png'))

In [ ]:
!zip -qr results.zip outputs artifacts
files.download('results.zip')